# FIR Assignment 01 — Solved + ES Concepts

**Setup note:** This notebook runs inside Docker. The ES host is `elasticsearch`, not `localhost`.
Data lives at `/work/data/` inside the container.

---

## Docker quick-start
```bash
# 1. host setting (Linux only, run once on your machine)
sudo sysctl -w vm.max_map_count=262144

# 2. start the stack
docker compose up -d --build

# 3. unzip data
unzip data01.zip -d data/

# 4. open Jupyter at http://localhost:8888
# 5. Kibana DevTools at http://localhost:5601
```

---
# PART 01 — Performance Evaluation

## ES Concept 0: Why metrics matter before you touch ES
MAP, recall@k, and P@10 are what your index design is optimised for.
You write them first so you can measure every ES experiment that follows.

In [ ]:
# Test data — same as the assignment
relevant  = set([2, 3, 5, 8, 13, 17, 21, 34, 38])
retrieved = [14, 4, 2, 18, 16, 8, 46, 32, 17, 34, 33, 22, 47, 39, 11]

In [ ]:
# ── Preparation: Success@k ────────────────────────────────────────────────────
# CONCEPT: set intersection. retrieved[:k] is a list slice.

def success_at_5(relevant, retrieved):
    # Any relevant doc in first 5?
    return 1 if len(set(retrieved[:5]) & relevant) > 0 else 0

def success_at_10(relevant, retrieved):
    return 1 if len(set(retrieved[:10]) & relevant) > 0 else 0

print('Success@5:', success_at_5(relevant, retrieved))   # expected: 1
print('Success@10:', success_at_10(relevant, retrieved)) # expected: 1

In [ ]:
# ── Exercise 01.A: Precision, Recall, F-measure ───────────────────────────────
# CONCEPT: set intersection gives true positives.
# Precision = TP / (TP + FP) = relevant_retrieved / total_retrieved
# Recall    = TP / (TP + FN) = relevant_retrieved / total_relevant

def precision(relevant, retrieved):
    if len(retrieved) == 0:
        return 0.0
    tp = len(set(retrieved) & relevant)
    return tp / len(retrieved)

def recall(relevant, retrieved):
    if len(relevant) == 0:
        return 0.0
    tp = len(set(retrieved) & relevant)
    return tp / len(relevant)

def f_measure(relevant, retrieved, beta=1):
    p = precision(relevant, retrieved)
    r = recall(relevant, retrieved)
    if p + r == 0:
        return 0.0
    return (1 + beta**2) * p * r / (beta**2 * p + r)

print(f'Precision: {precision(relevant, retrieved):.3f}')   # 5/15 = 0.333
print(f'Recall:    {recall(relevant, retrieved):.3f}')      # 5/9  = 0.556
print(f'F-measure: {f_measure(relevant, retrieved):.3f}')   # 0.417

In [ ]:
# ── Exercise 01.B: Precision@k and R-Precision ────────────────────────────────
# CONCEPT: ranked evaluation — we care about POSITION of relevant docs.
# P@k: precision considering only top-k results.
# R-Precision: P@R where R = number of relevant documents.
#   Special because it normalises across topics with different numbers of relevant docs.
#   This matters for your genomics corpus: some gene queries have 3 relevant docs, some 40.

def precision_at_k(relevant, retrieved, k):
    top_k = retrieved[:k]
    tp = len(set(top_k) & relevant)
    return tp / k

def r_precision(relevant, retrieved):
    R = len(relevant)
    return precision_at_k(relevant, retrieved, k=R)

print('P@1:  %1.2f' % precision_at_k(relevant, retrieved, k=1))  # 0.00
print('P@5:  %1.2f' % precision_at_k(relevant, retrieved, k=5))  # 0.20
print('P@10: %1.2f' % precision_at_k(relevant, retrieved, k=10)) # 0.30
print('R-P:  %1.2f' % r_precision(relevant, retrieved))           # 0.22

In [ ]:
# ── Exercise 01.D: Interpolated precision at recall X ────────────────────────
# CONCEPT: the 11-point precision-recall curve.
# For each recall level X (0.0, 0.1, ... 1.0), find the MAXIMUM precision
# achievable at any rank where recall >= X. This smooths the curve.
# Formula 8.7 from Manning et al.

def interpolated_precision_at_recall_X(relevant, retrieved, X):
    max_precision = 0.0
    for k in range(1, len(retrieved) + 1):
        r = recall(relevant, retrieved[:k])
        if r >= X:
            p = precision(relevant, retrieved[:k])
            max_precision = max(max_precision, p)
    return max_precision

for x in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    print('Pr_i@Re%.0f: %1.2f' % (x*10, interpolated_precision_at_recall_X(relevant, retrieved, X=x)))

In [ ]:
# ── Exercise 01.D: Average Precision ─────────────────────────────────────────
# CONCEPT: summarise the precision-recall curve in ONE number per query.
# For each relevant doc retrieved, compute precision at that rank.
# Average those precision values.
# MAP = mean of AP across all queries.
# THIS IS YOUR PRIMARY METRIC for the course project.

def average_precision(relevant, retrieved):
    if len(relevant) == 0:
        return 0.0
    ap = 0.0
    relevant_found = 0
    for k, doc in enumerate(retrieved, start=1):
        if doc in relevant:
            relevant_found += 1
            ap += relevant_found / k   # precision at rank k
    return ap / len(relevant)

print('AP: %.4f' % average_precision(relevant, retrieved))  # ~0.3275

In [ ]:
# ── TREC file readers (provided by course, unchanged) ─────────────────────────
def read_qrels_file(qrels_file):
    trec_relevant = dict()
    with open(qrels_file, 'r') as qrels:
        for line in qrels:
            (qid, q0, doc_id, rel) = line.strip().split()
            if qid not in trec_relevant:
                trec_relevant[qid] = set()
            if rel == '1':
                trec_relevant[qid].add(doc_id)
    return trec_relevant

def read_run_file(run_file):
    trec_retrieved = dict()
    with open(run_file, 'r') as run:
        for line in run:
            (qid, q0, doc_id, rank, score, tag) = line.strip().split()
            if qid not in trec_retrieved:
                trec_retrieved[qid] = []
            trec_retrieved[qid].append(doc_id)
    return trec_retrieved

def read_eval_files(qrels_file, run_file):
    return read_qrels_file(qrels_file), read_run_file(run_file)

(all_relevant, all_retrieved) = read_eval_files(
    '/work/data/FIR-s05-training-qrels.txt',
    '/work/data/baseline.run'
)
print('Queries loaded:', len(all_relevant))

In [ ]:
# ── Exercise 01.E: Mean Average Precision ────────────────────────────────────
# MAP = mean of AP over all 50 topics.
# This is the single number you compare across systems in your project.

def mean_average_precision(all_relevant, all_retrieved):
    total = 0
    count = 0
    for qid in all_relevant:
        relevant  = all_relevant[qid]
        retrieved = all_retrieved.get(qid, [])
        total += average_precision(relevant, retrieved)
        count += 1
    return total / count

mapr = mean_average_precision(all_relevant, all_retrieved)
print('MAP (baseline): %.4f' % mapr)

In [ ]:
# ── mean_metric + trec_eval (provided, unchanged) ────────────────────────────
def mean_metric(measure, all_relevant, all_retrieved):
    total = 0
    count = 0
    for qid in all_relevant:
        relevant  = all_relevant[qid]
        retrieved = all_retrieved.get(qid, [])
        total += measure(relevant, retrieved)
        count += 1
    return 'mean ' + measure.__name__, total / count

def trec_eval(qrels_file, run_file):
    def precision_at_1(rel, ret):  return precision_at_k(rel, ret, k=1)
    def precision_at_5(rel, ret):  return precision_at_k(rel, ret, k=5)
    def precision_at_10(rel, ret): return precision_at_k(rel, ret, k=10)
    def precision_at_50(rel, ret): return precision_at_k(rel, ret, k=50)
    def precision_at_100(rel, ret):return precision_at_k(rel, ret, k=100)
    def precision_at_recall_00(rel, ret): return interpolated_precision_at_recall_X(rel, ret, X=0.0)
    def precision_at_recall_05(rel, ret): return interpolated_precision_at_recall_X(rel, ret, X=0.5)
    def precision_at_recall_10(rel, ret): return interpolated_precision_at_recall_X(rel, ret, X=1.0)

    (all_rel, all_ret) = read_eval_files(qrels_file, run_file)
    metrics = [
        success_at_5, success_at_10,
        r_precision, precision_at_1, precision_at_5, precision_at_10,
        precision_at_50, precision_at_100,
        precision_at_recall_00, precision_at_recall_05, precision_at_recall_10,
        average_precision
    ]
    return [mean_metric(m, all_rel, all_ret) for m in metrics]

def print_trec_eval(qrels_file, run_file):
    print('Results for', run_file)
    for (metric, score) in trec_eval(qrels_file, run_file):
        print('{:<35} {:.4f}'.format(metric, score))

print_trec_eval(
    '/work/data/FIR-s05-training-qrels.txt',
    '/work/data/baseline.run'
)

In [ ]:
# ── Exercise 01.F: Sign test ──────────────────────────────────────────────────
# CONCEPT: significance testing.
# For each query, which system scores higher?
# better = count queries where system2 > system1
# worse  = count queries where system2 < system1
# p-value from binomial distribution: both directions (two-tailed).
# If p < 0.05 → the difference is statistically significant.
# For your PROJECT use the permutation test (stronger than sign test).

from scipy.stats import binom

def sign_test_values(measure, qrels_file, run_file_1, run_file_2):
    all_relevant    = read_qrels_file(qrels_file)
    all_retrieved_1 = read_run_file(run_file_1)
    all_retrieved_2 = read_run_file(run_file_2)
    better = 0
    worse  = 0
    for qid in all_relevant:
        rel = all_relevant[qid]
        s1  = measure(rel, all_retrieved_1.get(qid, []))
        s2  = measure(rel, all_retrieved_2.get(qid, []))
        if s2 > s1:  better += 1
        elif s2 < s1: worse  += 1
    return (better, worse)

def precision_at_rank_5(rel, ret):
    return precision_at_k(rel, ret, k=5)

(better, worse) = sign_test_values(
    precision_at_rank_5,
    '/work/data/trec.qrels',
    '/work/data/utbase.run',
    '/work/data/utexact.run'
)
n = better + worse
# two-tailed p-value: probability of observing this imbalance by chance
p_value = 2 * binom.cdf(min(better, worse), n, 0.5)
print(f'Better: {better}, Worse: {worse}, n: {n}')
print(f'Two-tailed p-value: {p_value:.4f}')
print('Significant at p<0.05:', p_value < 0.05)

---
# PART 02 — Indexing and Querying with Elasticsearch

## ES Concept 1: The inverted index
ES stores your documents in an **inverted index**: a mapping from every token
to the list of documents containing it, along with term frequency.
This is what makes `match` queries fast — ES only scores documents in the
posting list for your query terms, not every document.

## ES Concept 2: Analyzers — what goes INTO the index
An analyzer is a pipeline: `char_filter → tokenizer → token_filters`.
The same analyzer runs at index time AND query time (usually).
This is why `molecule` and `molecular` behave differently without a stemmer.

## ES Concept 3: Single shard for experiments
`number_of_shards: 1` ensures IDF is computed globally, not per-shard.
Critical for reproducible relevance scores in a research setting.

In [ ]:
import elasticsearch
import elasticsearch.helpers
import json
from pprint import pprint

# ── Connect to ES (Docker host = 'elasticsearch', not 'localhost') ────────────
# ES Concept 4: The REST API
# ES exposes everything via HTTP. The Python client wraps those HTTP calls.
# http://elasticsearch:9200 is the cluster endpoint.
es = elasticsearch.Elasticsearch(host='elasticsearch')

# Verify connection
print(es.info()['version']['number'])

In [ ]:
# ── Bulk indexing helpers (provided by course, adapted for Docker paths) ──────
# ES Concept 5: Bulk API
# Instead of one HTTP request per document, bulk sends batches.
# chunk_size=2000 means 2000 docs per HTTP request → much faster.
# streaming_bulk is memory-efficient — one doc at a time in Python.

def read_documents(file_name):
    with open(file_name, 'r') as documents:
        for line in documents:
            doc_line = json.loads(line)
            if 'index' in doc_line:
                id = doc_line['index']['_id']
            elif 'PMID' in doc_line:
                doc_line['_id'] = id
                yield doc_line
            else:
                raise ValueError('Error in index file')

def create_index(es, index_name, body={}):
    es.indices.delete(index=index_name, ignore=[400, 404])
    es.indices.create(index=index_name, body=body)

def index_documents(es, collection_file_name, index_name, body={}):
    create_index(es, index_name, body)
    return elasticsearch.helpers.bulk(
        es,
        read_documents(collection_file_name),
        index=index_name,
        chunk_size=2000,
        request_timeout=30
    )

# Index with default settings (BM25, standard analyzer)
# Note: PMID is used as _id for matching with qrels
index_documents(es, '/work/data/FIR-s05-medline.json', 'genomics-base', body={})
print('Indexed. Count:', es.count(index='genomics-base')['count'])

In [ ]:
# ── Exercise 02.A: Index properties and querying ──────────────────────────────

# ES Concept 6: _count API — fast document counting without scoring

# 1. Total documents in index
total = es.count(index='genomics-base')['count']
print(f'Total docs: {total}')

# 2. Documents containing 'molecule'
# ES Concept 7: match query — analyzed at query time (same analyzer as index)
body = {'query': {'match': {'_all': 'molecule'}}}
# use query_string for multi-field search across all fields
body2 = {'track_total_hits': True, 'query': {'query_string': {'query': 'molecule'}}}
r = es.count(index='genomics-base', body=body2)
print(f'Docs with molecule: {r["count"]}')

# 3. Documents containing 'molecular'
body3 = {'track_total_hits': True, 'query': {'query_string': {'query': 'molecular'}}}
r3 = es.count(index='genomics-base', body=body3)
print(f'Docs with molecular: {r3["count"]}')
# Note: different counts — no stemming yet. This motivates Exercise 03.B.

# 4. Documents containing 'cell' AND 'blood'
# ES Concept 8: bool query — must/should/filter clauses
body4 = {'track_total_hits': True, 'query': {'query_string': {
    'query': 'blood AND cell', 'default_operator': 'AND'
}}}
r4 = es.count(index='genomics-base', body=body4)
print(f'Docs with cell AND blood: {r4["count"]}')

In [ ]:
# ── Exercise 02.B: Python ES library search ───────────────────────────────────
# ES Concept 9: search() returns hits with _score (BM25 relevance score)

body = {
    'track_total_hits': True,
    'query': {'query_string': {'query': 'molecule'}},
    'size': 5
}
response = es.search(index='genomics-base', body=body)
print('Total hits:', response['hits']['total']['value'])
print('\nTop 5 titles:')
for hit in response['hits']['hits']:
    print(f"  [{hit['_score']:.2f}] {hit['_source'].get('TI', 'No title')[:80]}")

In [ ]:
# ── Exercise 02.C: Elasticsearch DSL ─────────────────────────────────────────
# ES Concept 10: DSL is syntactic sugar over the same JSON query API

from elasticsearch_dsl import Search, Q

# 1. Search for 'molecule'
s = Search(using=es, index='genomics-base')
s = s.query('query_string', query='molecule')
s = s.extra(track_total_hits=True)
response = s.execute()
print('DSL molecule hits:', response.hits.total.value)

# 2. Search for 'cell' AND 'blood'
s2 = Search(using=es, index='genomics-base')
s2 = s2.query('query_string', query='cell AND blood', default_operator='AND')
s2 = s2.extra(track_total_hits=True)
response2 = s2.execute()
print('DSL cell AND blood hits:', response2.hits.total.value)

In [ ]:
# ── Exercise 02.D: Make a TREC run ───────────────────────────────────────────
# ES Concept 11: multi_match query — search across multiple fields simultaneously
# TI (title) and AB (abstract) are your text fields.
# IMPORTANT: use PMID as doc_id in the run, NOT _id — qrels use PMID.

def make_trec_run(es, topics_file_name, run_file_name, index_name='genomics', run_name='test'):
    with open(run_file_name, 'w') as run_file:
        with open(topics_file_name, 'r') as test_queries:
            for line in test_queries:
                (qid, query) = line.strip().split('\t')
                
                # ES Concept 12: multi_match best_fields
                # Searches TI and AB, takes the best-scoring field per doc.
                # TI^2 = title boost (title match is stronger signal than abstract match)
                body = {
                    'query': {
                        'multi_match': {
                            'query':  query,
                            'fields': ['TI^2', 'AB'],
                            'type':   'best_fields'
                        }
                    },
                    'size': 1000  # TREC allows up to 1000 per topic
                }
                
                # ES Concept 13: dfs_query_then_fetch
                # Collects term stats globally before scoring.
                # Makes IDF consistent across shards — critical for experiments.
                result = es.search(
                    index=index_name,
                    body=body,
                    search_type='dfs_query_then_fetch'
                )
                
                for rank, hit in enumerate(result['hits']['hits'], start=1):
                    doc_id = hit['_source'].get('PMID', hit['_id'])  # use PMID not _id
                    score  = hit['_score']
                    run_file.write(f'{qid} Q0 {doc_id} {rank} {score:.6f} {run_name}\n')

make_trec_run(
    es,
    '/work/data/FIR-s05-training-queries-simple.txt',
    '/work/runs/baseline_nb.run',
    'genomics-base',
    run_name='test01'
)
print('Run written. Evaluating...')
print_trec_eval(
    '/work/data/FIR-s05-training-qrels.txt',
    '/work/runs/baseline_nb.run'
)

---
# PART 03 — Tokenization and Analyzers

## ES Concept 14: The _analyze API — your debugging tool
Before indexing, always check what tokens your analyzer produces.
A token that never reaches the index can never match a query.

## ES Concept 15: Custom analyzer pipeline
`char_filter → tokenizer → token_filter_1 → token_filter_2 → ...`
Each stage transforms the token stream.
The synonym_graph filter is the key one for your project.

In [ ]:
# ── Exercise 03.A: Custom chat-language analyzer ──────────────────────────────
# This teaches the full analyzer pipeline structure.
# For your PROJECT, replace chat abbreviations with gene aliases.

tweet_analyzer = {
    'settings': {
        'number_of_shards': 1,
        'analysis': {
            'filter': {
                # ES Concept 16: synonym filter — maps tokens to expanded forms
                # Format: 'source => target' (one-way) or 'a, b, c' (all equivalent)
                # This is the SAME mechanism used for gene alias expansion!
                'chat_synonyms': {
                    'type': 'synonym_graph',
                    'synonyms': [
                        'b4 => before',
                        'abt => about',
                        'chk => check',
                        'dm => direct message',
                        'f2f => face-to-face'
                    ]
                },
                'english_stop': {
                    'type': 'stop',
                    'stopwords': '_english_'
                }
            },
            'analyzer': {
                'chat_analyzer': {
                    'type':      'custom',
                    'tokenizer': 'standard',
                    'filter': [
                        'chat_synonyms',    # expand abbreviations first
                        'lowercase',        # then lowercase
                        'english_stop'      # then remove stopwords
                    ]
                }
            }
        }
    },
    'mappings': {
        'properties': {
            'all': {
                'type':     'text',
                'analyzer': 'chat_analyzer'
            }
        }
    }
}

create_index(es, 'chat-test', body=tweet_analyzer)

# Test what the analyzer produces
body = {'field': 'all', 'text': 'done it b4! what abt dm me?'}
tokens = es.indices.analyze(index='chat-test', body=body)
print('Input: "done it b4! what abt dm me?"')
print('Tokens:', [t['token'] for t in tokens['tokens']])
# Expected: ['done', 'b4', 'before', 'abt', 'about', 'direct', 'message']
# Note: stopwords (it, what, me) are removed

In [ ]:
# ── Exercise 03.B: Stemmers ───────────────────────────────────────────────────
# ES Concept 17: Stemming — reduces words to their root form
# 'molecule', 'molecules', 'molecular' → 'molecul' (Porter)
# This closes the vocabulary gap for morphological variants.
# For gene names (BCL2, STAT5A), stemming is DANGEROUS — don't stem identifiers.
# Solution for your project: separate fields (TI/AB with stemming, gene_symbols without)

stemmer_body = {
    'settings': {
        'number_of_shards': 1,
        'analysis': {
            'filter': {
                'english_stemmer': {
                    'type':     'stemmer',
                    'language': 'english'
                },
                'english_stop': {
                    'type':      'stop',
                    'stopwords': '_english_'
                }
            },
            'analyzer': {
                'english_stem': {
                    'type':      'custom',
                    'tokenizer': 'standard',
                    'filter': ['lowercase', 'english_stop', 'english_stemmer']
                }
            }
        }
    },
    'mappings': {
        'properties': {
            'TI': {'type': 'text', 'analyzer': 'english_stem'},
            'AB': {'type': 'text', 'analyzer': 'english_stem'}
        }
    }
}

# Show what stemming does to 'molecule' vs 'molecular'
# We can test without reindexing by creating a temporary index
create_index(es, 'stem-test', body=stemmer_body)

for word in ['molecule', 'molecular', 'molecules', 'BCL2', 'STAT5A', 'kinase', 'kinases']:
    tokens = es.indices.analyze(index='stem-test', body={'analyzer': 'english_stem', 'text': word})
    stems  = [t['token'] for t in tokens['tokens']]
    print(f'  {word:20} → {stems}')

In [ ]:
# Now index the full collection with stemming and compare
print('Indexing with stemmer (takes ~5min)...')
index_documents(es, '/work/data/FIR-s05-medline.json', 'genomics-stem', stemmer_body)

# Compare molecule vs molecular — both should now return same docs
for term in ['molecule', 'molecular']:
    r = es.count(index='genomics-stem', body={'query': {'query_string': {'query': term}}})
    print(f'  genomics-stem  |  {term:15} → {r["count"]} docs')

# Compare with no-stemmer index
for term in ['molecule', 'molecular']:
    r = es.count(index='genomics-base', body={'query': {'query_string': {'query': term}}})
    print(f'  genomics-base  |  {term:15} → {r["count"]} docs')

# RESULT: 'molecule' and 'molecular' should now return MORE similar counts
# because both stem to 'molecul' and hit the same posting lists.
# 'molecular' gains more because it's the morphological variant.

---
# PART 04 — Search Models

## ES Concept 18: Similarity modules
ES supports pluggable similarity functions:
- `BM25` (default): probabilistic, handles TF saturation and doc length normalization
- `boolean`: pure 0/1 match — used for exact-match retrieval
- `LMJelinekMercer`: language model with linear interpolation smoothing
- `LMDirichlet`: language model with Bayesian smoothing (mu controls prior)

## ES Concept 19: copy_to — virtual compound field
`copy_to: 'title-abstract'` copies tokens from TI and AB into one combined field.
You search the combined field without duplicating stored data.

In [ ]:
# ── Boolean similarity ────────────────────────────────────────────────────────
# Score = 1 if term matches, 0 if not. No TF, no IDF, no length norm.
# Use case: exact match, faceted filtering.

boolean = {
    'settings': {'number_of_shards': 1},
    'mappings': {
        'properties': {
            'AB': {'type': 'text', 'copy_to': 'title-abstract', 'similarity': 'boolean'},
            'TI': {'type': 'text', 'copy_to': 'title-abstract', 'similarity': 'boolean'},
            'title-abstract': {'type': 'text', 'similarity': 'boolean'}
        }
    }
}

print('Indexing boolean index...')
index_documents(es, '/work/data/FIR-s05-medline.json', 'genomics-bool', body=boolean)
make_trec_run(es, '/work/data/FIR-s05-training-queries-simple.txt',
              '/work/runs/boolean.run', 'genomics-bool', 'bool01')
print_trec_eval('/work/data/FIR-s05-training-qrels.txt', '/work/runs/boolean.run')

In [ ]:
# ── Exercise 04.B: Language Models ───────────────────────────────────────────
# ES Concept 20: Language Model smoothing
# LM ranks by P(query | document).
# Without smoothing, unseen terms get P=0 → zero score.
# Jelinek-Mercer: interpolate with collection model (lambda=0.2 = 20% collection)
# Dirichlet: Bayesian prior (mu=2000 = prior document length)

lmjelinekmercer = {
    'settings': {
        'number_of_shards': 1,
        'similarity': {
            'lm_jm': {
                'type':   'LMJelinekMercer',
                'lambda': 0.2
            }
        }
    },
    'mappings': {
        'properties': {
            'AB': {'type': 'text', 'copy_to': 'all', 'similarity': 'lm_jm'},
            'TI': {'type': 'text', 'copy_to': 'all', 'similarity': 'lm_jm'},
            'all': {'type': 'text', 'similarity': 'lm_jm'}
        }
    }
}

print('Indexing LM Jelinek-Mercer...')
index_documents(es, '/work/data/FIR-s05-medline.json', 'genomics-jm', lmjelinekmercer)
make_trec_run(es, '/work/data/FIR-s05-training-queries-simple.txt',
              '/work/runs/lmjelinekmercer.run', 'genomics-jm', 'jm01')

In [ ]:
dirichlet = {
    'settings': {
        'number_of_shards': 1,
        'similarity': {
            'lm_dir': {
                'type': 'LMDirichlet',
                'mu':   2000
            }
        }
    },
    'mappings': {
        'properties': {
            'AB': {'type': 'text', 'copy_to': 'all', 'similarity': 'lm_dir'},
            'TI': {'type': 'text', 'copy_to': 'all', 'similarity': 'lm_dir'},
            'all': {'type': 'text', 'similarity': 'lm_dir'}
        }
    }
}

print('Indexing LM Dirichlet...')
index_documents(es, '/work/data/FIR-s05-medline.json', 'genomics-dirichlet', dirichlet)
make_trec_run(es, '/work/data/FIR-s05-training-queries-simple.txt',
              '/work/runs/dirichlet.run', 'genomics-dirichlet', 'dir01')

In [ ]:
# ── Exercise 04.C: Model comparison ──────────────────────────────────────────
# ES Concept 21: _explain API — the debugging superpower
# Shows EXACTLY how a score was computed: which terms matched, IDF, TF, field norm.
# Use this in your project to diagnose why a relevant doc ranked low.

print('=== BASELINE (BM25 default) ===')
print_trec_eval('/work/data/FIR-s05-training-qrels.txt', '/work/data/baseline.run')

print('\n=== BOOLEAN ===')
print_trec_eval('/work/data/FIR-s05-training-qrels.txt', '/work/runs/boolean.run')

print('\n=== LM JELINEK-MERCER (lambda=0.2) ===')
print_trec_eval('/work/data/FIR-s05-training-qrels.txt', '/work/runs/lmjelinekmercer.run')

print('\n=== LM DIRICHLET (mu=2000) ===')
print_trec_eval('/work/data/FIR-s05-training-qrels.txt', '/work/runs/dirichlet.run')

In [ ]:
# ── _explain API demo ─────────────────────────────────────────────────────────
# Show the BM25 score breakdown for a specific query + doc pair.
# This is how you diagnose: why did doc X rank where it did?

query_body = {'query': {'match': {'TI': 'structure refinement'}}}
explanation = es.explain(index='genomics-base', id='3', body=query_body)
print('Score:', explanation['explanation']['value'])
print('\nBreakdown:')
pprint(explanation['explanation'])

In [ ]:
# ── BONUS: BM25 from scratch ──────────────────────────────────────────────────
# Implement BM25 using term vectors — understand what ES computes.
# k1=1.2 (TF saturation), b=0.75 (length normalization) are ES defaults.

import math

def bm25_similarity(query, doc_id, index='genomics-base', field='TI', k1=1.2, b=0.75):
    # get query tokens (analyzed the same way as at index time)
    query_tokens = es.indices.analyze(index=index, body={'field': field, 'text': query})
    tokens = [t['token'] for t in query_tokens['tokens']]

    # get term vector + field stats for this document
    tv = es.termvectors(
        index=index, id=doc_id, fields=field,
        term_statistics=True, field_statistics=True, offsets=False
    )

    if field not in tv['term_vectors']:
        return 0.0

    vector  = tv['term_vectors'][field]['terms']
    fstats  = tv['term_vectors'][field]['field_statistics']
    N       = fstats['doc_count']           # total docs in index
    avgdl   = fstats['sum_ttf'] / N         # average doc length
    dl      = sum(t.get('term_freq', 0) for t in vector.values())  # this doc's length

    similarity = 0.0
    for token in tokens:
        if token not in vector:
            continue
        tf  = vector[token]['term_freq']
        df  = vector[token]['doc_freq']      # docs containing this term
        idf = math.log(1 + (N - df + 0.5) / (df + 0.5))  # BM25 IDF
        tf_norm = tf * (k1 + 1) / (tf + k1 * (1 - b + b * dl / avgdl))
        similarity += idf * tf_norm

    return similarity

score = bm25_similarity('structure refinement', '3')
print(f'Our BM25:  {score:.4f}')
# ES score from _explain:
explain = es.explain(index='genomics-base', id='3',
                     body={'query': {'match': {'TI': 'structure refinement'}}})
print(f'ES BM25:   {explain["explanation"]["value"]:.4f}')
print('Note: small difference due to ES Lucene implementation details (IDF formula variant)')

---
# ES Concepts Summary

| Concept | What it is | Where you used it |
|---------|-----------|-------------------|
| Inverted index | token → posting list | every index you created |
| Analyzer pipeline | char_filter → tokenizer → token_filter | 03.A, 03.B |
| _analyze API | inspect tokens before indexing | 03.A debug |
| synonym_graph filter | alias expansion in the index | 03.A, your project |
| Stemmer filter | morphological normalisation | 03.B |
| single shard | consistent IDF for experiments | all indices |
| BM25 similarity | default scoring (k1, b) | baseline |
| boolean similarity | exact match, no weighting | 04.A |
| LM similarities | probabilistic language models | 04.B |
| copy_to field | virtual compound field | 04.A, 04.B |
| multi_match | search multiple fields at once | 02.D |
| dfs_query_then_fetch | global IDF across shards | 02.D |
| _explain API | score breakdown for debugging | bonus |
| bulk / streaming_bulk | efficient large-scale indexing | all indexing |
| bool query (must/should/filter) | compound query logic | your project |
| function_score | inject authority/recency signals | your project |
| collapse | MaxP passage aggregation | your project |
